# Task 2 — PCA + KNN

**Requirement:** implement PCA for dimensionality reduction, classify with KNN
(`n_neighbors=2`), and report Macro-F1 for **2000, 1000, 500 and 100** components.

sklearn is permitted for this task.

**Note on "on the test set":** the Kaggle test labels are hidden, so Macro-F1
cannot be computed there. We score a held-out 15% validation split carved from
the training set — the standard substitute. This is stated in the report.


In [ ]:
from pathlib import Path
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import f1_score, classification_report

DATA = Path("..") / "data"
OUT = Path("outputs"); OUT.mkdir(parents=True, exist_ok=True)
SEED = 42


## 1. Load the provided features

`train_features.csv` is laid out as `[id, label, 0001 ... 5000]` — it carries
the **label as well as the id**.

Dropping only the id column would leave `label` sitting in `X` as a feature.
That is target leakage: the model reads the answer straight off column 2, scores
near-perfectly on validation, and collapses on the leaderboard. We take the
feature columns as the *intersection with the test set*, which cannot contain
the label by construction.


In [ ]:
train = pd.read_csv(DATA / "train_features.csv")
test  = pd.read_csv(DATA / "test_features.csv")
print("raw shapes:", train.shape, test.shape)

y = train["label"].to_numpy().astype(int)
test_ids = test["id"].to_numpy()

id_cols = ["id"]
feature_cols = [c for c in train.columns if c in set(test.columns) and c not in id_cols]
dropped = [c for c in train.columns if c not in feature_cols and c not in id_cols]
print("dropped non-feature columns:", dropped)      # -> ['label']

X  = train[feature_cols].to_numpy(np.float64)
Xt = test[feature_cols].to_numpy(np.float64)
print("X:", X.shape, "| X_test:", Xt.shape, "| positive rate:", round(y.mean(), 4))


## 2. Train / validation split

PCA is fitted on the **training split only** and applied to validation via
`transform`. Fitting it on all the data first would leak the validation set's
covariance structure into the projection.


In [ ]:
rng = np.random.default_rng(SEED)
idx = rng.permutation(len(X))
n_val = int(0.15 * len(X))
val_idx, tr_idx = idx[:n_val], idx[n_val:]

X_tr, y_tr = X[tr_idx], y[tr_idx]
X_va, y_va = X[val_idx], y[val_idx]
print("train:", X_tr.shape, "| val:", X_va.shape)

baseline = f1_score(y_va, np.ones_like(y_va), average="macro")
print(f"always-predict-majority baseline Macro-F1 = {baseline:.4f}")


## 3. Fit PCA once, then slice

Principal components come out **ordered by explained variance**, so the first
`n` columns of a 2000-component projection are *exactly* the `n`-component
projection. Fitting four separate PCAs would repeat the same expensive SVD for
identical numbers.


In [ ]:
t0 = time.time()
pca = PCA(n_components=2000, random_state=SEED)
A = pca.fit_transform(X_tr)     # train projected
B = pca.transform(X_va)         # validation projected
print(f"PCA fitted in {time.time()-t0:.0f}s | "
      f"total explained variance = {pca.explained_variance_ratio_.sum():.4f}")


## 4. Required sweep — KNN with `n_neighbors=2`

In [ ]:
rows = []
for n in (2000, 1000, 500, 100):
    knn = KNeighborsClassifier(n_neighbors=2).fit(A[:, :n], y_tr)
    pred = knn.predict(B[:, :n])
    rows.append({
        "n_components": n,
        "macro_f1": f1_score(y_va, pred, average="macro"),
        "explained_variance": float(pca.explained_variance_ratio_[:n].sum()),
    })
    print(f"n_components={n:5d} | Macro-F1={rows[-1]['macro_f1']:.4f} "
          f"| explained_var={rows[-1]['explained_variance']:.4f}")

report = pd.DataFrame(rows)
report.to_csv(OUT / "task2_pca_knn_report_table.csv", index=False)
report


### The trend

Macro-F1 **rises as components fall**, while explained variance collapses.
Going 2000 -> 100 discards ~62 percentage points of variance and *gains* about
0.22 Macro-F1. Retaining variance and retaining useful signal are not the same
thing.

The mechanism is **distance concentration**. KNN classifies by Euclidean
distance, and in high dimensions the ratio between the nearest and the farthest
neighbour tends to 1 — every point is roughly equidistant from every other, so
"nearest" stops meaning "similar". Cutting to 100 dimensions restores meaningful
distances. The trailing components of a TF-IDF matrix are mostly noise
directions from rare terms, so discarding them removes noise rather than signal.


In [ ]:
plt.figure(figsize=(6, 4))
plt.plot([str(r["n_components"]) for r in rows], [r["macro_f1"] for r in rows], marker="o")
plt.axhline(baseline, ls="--", c="grey", lw=1, label=f"majority baseline ({baseline:.3f})")
plt.xlabel("Number of PCA components"); plt.ylabel("Macro-F1")
plt.title("KNN (k=2) Macro-F1 vs PCA components")
plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
plt.savefig(OUT / "task2_f1_vs_components.png", dpi=150)
plt.show()


## 5. Submission

Built from the best setting found above.


In [ ]:
best = max(rows, key=lambda r: r["macro_f1"])
n_best = best["n_components"]
print("best setting:", n_best, "components, Macro-F1", round(best["macro_f1"], 4))

knn = KNeighborsClassifier(n_neighbors=2).fit(A[:, :n_best], y_tr)
pred_test = knn.predict(pca.transform(Xt)[:, :n_best])

sub = pd.DataFrame({"id": test_ids, "label": pred_test})
sub.to_csv(OUT / "PCA_KNN_predictions.csv", index=False)
print("saved outputs/PCA_KNN_predictions.csv | predicted positive rate:",
      round(pred_test.mean(), 4))


## 6. Supplementary — two things the required grid hides

Run `python3 supplementary_analysis.py` for the full version. Summary:

**(a) The optimum is exactly at 100 components.** Extending the sweep below 100
shows performance *falls* again (50: 0.6457, 20: 0.6400, 10: 0.6141, 5: 0.5907,
2: 0.5368). The true shape is an **inverted U**, and the mandated grid stops
precisely at the peak — which makes the four required points look monotonic when
they are really the left arm of a curve. This is the bias-variance tradeoff in
dimensionality: too many dimensions and distances are meaningless, too few and
the classes are no longer separable.

**(b) `n_neighbors=2` is the worst k tested, at every component count.**

| n_components | k=2 | k=5 | k=15 | k=31 |
|---|---|---|---|---|
| 2000 | 0.4259 | 0.4646 | 0.4962 | **0.6060** |
| 1000 | 0.4962 | 0.5674 | 0.5722 | **0.5961** |
| 500  | 0.5733 | **0.6277** | 0.6205 | 0.6201 |
| 100  | 0.6511 | 0.6700 | 0.6789 | **0.6821** |

At 2000 components, moving k=2 -> k=31 is worth **+0.18 Macro-F1** — a bigger
effect than dimensionality reduction achieves anywhere in that region. k=2 is
pathological on a binary problem: a one-vote-each split is common and must be
broken arbitrarily, and a single atypical training point flips half the vote.

The brief mandates k=2, so the required table above uses it. The sensitivity is
worth reporting as a limitation of the prescribed setup, not of KNN itself.
